In [1]:
import $ivy.`org.apache.spark::spark-core:3.5.6`
import $ivy.`org.apache.spark::spark-sql:3.5.6`

import $ivy.$
import $ivy.$

In [2]:
import scala.util.{Try, Success, Failure}
import java.sql.Timestamp
import java.time.LocalDateTime
import java.time.format.DateTimeFormatter

object ParserFunction extends Serializable {
    private val formatter = DateTimeFormatter.ISO_LOCAL_DATE_TIME

    def parseBoolean(s: String): Option[Boolean] = {
        if (s == null || s.trim.isEmpty) None
        else Some(s.trim.toUpperCase == "Y")
    }
    def parseBigDecimal(s:String):Option[BigDecimal] = 
        Option(s).map(_.trim).filter(_.nonEmpty).flatMap(v => Try(BigDecimal(v)).toOption)
    def parseInt(s:String):Option[Int] = 
        Option(s).map(_.trim).filter(_.nonEmpty).flatMap(v => Try(v.toInt).toOption)
    def parseString(s:String):Option[String] = Option(s).map(_.trim)
    def parseQuotedString(s: String): Option[String] = {
        if (s == null || s.trim.isEmpty) {
        None
        } else {
        val trimmed = s.trim
        val clean = trimmed.stripPrefix("\"").stripSuffix("\"")
        Some(clean)
        }
    }

    def parseTimestamp(s: String): Option[Timestamp] = {
        if (s == null || s.trim.isEmpty) None
        else Try(Timestamp.valueOf(LocalDateTime.parse(s, formatter))).toOption
    }
}

import scala.util.{Try, Success, Failure}
import java.sql.Timestamp
import java.time.LocalDateTime
import java.time.format.DateTimeFormatter
defined object ParserFunction

In [3]:
import scala.util.{Try, Success, Failure}
import org.apache.spark.rdd.RDD

object TripDomain extends Serializable {
  case class Trip(
      vendorID: Option[Int],
      tpepPickupDatetime: Option[java.sql.Timestamp], 
      tpepDropoffDatetime: Option[java.sql.Timestamp],
      passengerCount: Option[Int],
      tripDistance: Option[BigDecimal],   
      ratecodeID: Option[Int],
      storeAndFwdFlag: Option[Boolean],
      puLocationID: Option[Int],
      doLocationID: Option[Int],
      paymentType: Option[Int],
      fareAmount: Option[BigDecimal],
      extra: Option[BigDecimal],
      mtaTax: Option[BigDecimal],
      tipAmount: Option[BigDecimal],
      tollsAmount: Option[BigDecimal],
      improvementSurcharge: Option[BigDecimal],
      totalAmount: Option[BigDecimal],
      congestionSurcharge: Option[BigDecimal],
      airportFee: Option[BigDecimal]
  ) {
  override def toString: String = {
      f"""
        |--- Trip ---
        |tpepPickupDatetime        :  ${tpepPickupDatetime.getOrElse("N/A")}
        |tpepDropoffDatetime       :  ${tpepDropoffDatetime.getOrElse("N/A")}
        |tripDistance              :  ${tripDistance.getOrElse(0)} miles
        |fareAmount                :  ${fareAmount.getOrElse(0)}
        |totalAmount               :  ${totalAmount.getOrElse(0)}
        |passengerCount            :  ${passengerCount.getOrElse(0)}
        |""".stripMargin
    }
  }

  def parseTrip(line: Array[String]): Option[Trip] = {
    import ParserFunction._
    Try {
      Trip(
        vendorID            = parseInt(line(0)),
        tpepPickupDatetime  = parseTimestamp(line(1)),
        tpepDropoffDatetime = parseTimestamp(line(2)),
        passengerCount      = parseInt(line(3)),
        tripDistance        = parseBigDecimal(line(4)),
        ratecodeID          = parseInt(line(5)),
        storeAndFwdFlag     = parseBoolean(line(6)),
        puLocationID        = parseInt(line(7)),
        doLocationID        = parseInt(line(8)),
        paymentType         = parseInt(line(9)),
        fareAmount          = parseBigDecimal(line(10)),
        extra               = parseBigDecimal(line(11)),
        mtaTax              = parseBigDecimal(line(12)),
        tipAmount           = parseBigDecimal(line(13)),
        tollsAmount         = parseBigDecimal(line(14)),
        improvementSurcharge= parseBigDecimal(line(15)),
        totalAmount         = parseBigDecimal(line(16)),
        congestionSurcharge = if (line.length > 17) parseBigDecimal(line(17)) else None,
        airportFee          = if (line.length > 18) parseBigDecimal(line(18)) else None
      )
    }.toOption
  }

  def processRDD(rawRdd: RDD[String]): RDD[Trip] = {
    // first - это акшен-> лучше избегать, поэтому применен вариант mapPartitionsWithIndex
    // val tripDataHeader = rawRdd.first
    rawRdd
      // .filter(line => line != tripDataHeader)
      .mapPartitionsWithIndex { (idx, iter) => if (idx == 0) iter.drop(1) else iter }
      .map(line => line.split(",", -1))
      .flatMap(line => parseTrip(line).toIterable)
  }
}


import scala.util.{Try, Success, Failure}
import org.apache.spark.rdd.RDD
defined object TripDomain

In [4]:
import scala.util.{Try, Success, Failure}
import org.apache.spark.rdd.RDD

object ZoneDomain extends Serializable {
  case class Zone(
      locationID: Option[Int],
      borough: Option[String],
      zone: Option[String],
      serviceZone: Option[String]
  ) {
  override def toString: String = {
      f"""
        |--- Zone ---
        |locationID     :  ${locationID.getOrElse(0)}
        |borough        :  ${borough.getOrElse("N/A")}
        |zone           :  ${zone.getOrElse("N/A")}
        |serviceZone    :  ${serviceZone.getOrElse("N/A")}
        |""".stripMargin
    }
  }

  def parseZone(line: Array[String]): Option[Zone] = {
    import ParserFunction._
    Try {
      Zone(
        locationID          = parseInt(line(0)),
        borough             = parseQuotedString(line(1)),
        zone                = parseQuotedString(line(2)),
        serviceZone         = parseQuotedString(line(3))
      )
    }.toOption
  }

  def processRDD(rawRdd: RDD[String]): RDD[Zone] = {
    rawRdd
      .mapPartitionsWithIndex { (idx, iter) => if (idx == 0) iter.drop(1) else iter }
      .map(line => line.split(",", -1))
      .flatMap(line => parseZone(line).toIterable)
  }
}


import scala.util.{Try, Success, Failure}
import org.apache.spark.rdd.RDD
defined object ZoneDomain

In [ ]:
import org.apache.spark.broadcast.Broadcast
import org.apache.spark.rdd.RDD
import java.util.Calendar

object AnalysisUtils extends Serializable {

  private def mapTripToStats(
      trip: TripDomain.Trip, 
      zonesBc: Broadcast[Map[Int, String]]
  ): Iterable[((Int, String), Int)] = {
    
    val res = for {
      time <- trip.tpepPickupDatetime
      locId <- trip.puLocationID
    } yield {
      val cal = Calendar.getInstance()
      cal.setTime(time)
      val hourOfDay = cal.get(Calendar.HOUR_OF_DAY)
      val zoneName = zonesBc.value.getOrElse(locId, "Unknown Zone")
      ((hourOfDay, zoneName), 1)
    }
    res.toIterable
  }  

  def calculateHourlyStats(
      tripRdd: RDD[TripDomain.Trip],
      zonesBc: Broadcast[Map[Int, String]]
  ): RDD[((Int, String), Int)] = {
    
    tripRdd
      .flatMap(trip => mapTripToStats(trip, zonesBc))
      .reduceByKey(_ + _)
  }
}

import org.apache.spark.broadcast.Broadcast
import org.apache.spark.rdd.RDD
import java.util.Calendar
defined object AnalysisUtils

In [13]:
import org.apache.spark.{SparkConf, SparkContext}
import org.apache.spark.sql.SparkSession

// 1. стоп сессий
SparkSession.getActiveSession.foreach(_.stop())

// 2. Конфиг и создание сессий
val conf = new SparkConf()
  .setAppName("RDD")
  .setMaster("local[4]")
  .set("spark.driver.memory","16g")
  .set("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
  .registerKryoClasses(Array(classOf[TripDomain.Trip], classOf[ZoneDomain.Zone]))
  .set("spark.log.level", "WARN")

val spark = SparkSession.builder().config(conf).getOrCreate()
val sc = spark.sparkContext
println(s"Spark version: ${spark.version}")

// 3. Загрузка и Обработка
val tripDataRdd = TripDomain.processRDD(sc.textFile("tripdata.csv"))
// tripDataRdd.take(5).foreach(println)
// tripDataRdd.collect().takeRight(5).foreach(println)
val zoneDataRdd = ZoneDomain.processRDD(sc.textFile("taxi_zone_lookup.csv"))
// zoneDataRdd.take(5).foreach(println)

// 4. броадкаст(отправляем мапу на все узлы кластера)
val zonesMap = zoneDataRdd
  .flatMap(z => z.locationID.map(id => (id, z.zone.getOrElse("Unknown"))))
  .collect()
  .toMap
val zonesBc = sc.broadcast(zonesMap)

// 5. объединяем время посадки и зону(район)
val hourlyStats = AnalysisUtils.calculateHourlyStats(tripDataRdd, zonesBc)

// 6. Сохраняем статистику вызовов по часам в файл
val csvBody = hourlyStats
  .map{case ((hour, zone), count) =>
    val quotedZone = zone.replace("\"", "\"\"")
    s"""$hour,"$quotedZone",$count"""
  }
val csvHeader = sc.parallelize(Seq("hour,zone_name,count"))
val csvResult = csvHeader.union(csvBody)
csvResult.coalesce(1).saveAsTextFile("stats_by_hour.csv")

//7. вывод на экран
// hourlyStats.sortByKey().collect().foreach { case ((hour, zone), count) =>
//   println(f"Hour: $hour%02d | Zone: $zone%-30s | Counts: $count")
// }

Setting Spark log level to "WARN".


Spark version: 3.5.6


import org.apache.spark.{SparkConf, SparkContext}
import org.apache.spark.sql.SparkSession
conf: SparkConf = org.apache.spark.SparkConf@146255bd
spark: SparkSession = org.apache.spark.sql.SparkSession@576c7577
sc: SparkContext = org.apache.spark.SparkContext@64e2a9eb
tripDataRdd: RDD[TripDomain.Trip] = MapPartitionsRDD[4] at flatMap at cmd3.sc:73
zoneDataRdd: RDD[ZoneDomain.Zone] = MapPartitionsRDD[9] at flatMap at cmd4.sc:38
zonesMap: Map[Int, String] = Map(
  69 -> "East Concourse/Concourse Village",
  138 -> "LaGuardia Airport",
  101 -> "Glen Oaks",
  249 -> "West Village",
  234 -> "Union Sq",
  88 -> "Financial District South",
  170 -> "Murray Hill",
  115 -> "Grymes Hill/Clifton",
  217 -> "South Williamsburg",
  5 -> "Arden Heights",
  120 -> "Highbridge Park",
  247 -> "West Concourse",
  202 -> "Roosevelt Island",
  10 -> "Baisley Park",
  56 -> "Corona",
  142 -> "Lincoln Square East",
  153 -> "Marble Hill",
  174 -> "Norwood",
  185 -> "Pelham Parkway",
  42 -> "Central H